# BikeSQL 数据查询与准备

本 notebook 用于对数据库进行常用查询、统计与数据质量检查。


In [4]:
import os
import sys
import pandas as pd
import sqlalchemy as sqla

# 读取配置
sys.path.append(os.getcwd())
try:
    import config
    print("config checked", flush=True)
except ImportError:
    print("Error: config.py not found,pls make one at first.")
    raise

DB_NAME = getattr(config, "DB_NAME", "COMP30830_SW")

engine = sqla.create_engine(
    f"mysql+pymysql://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}/{DB_NAME}"
    f"?charset=utf8mb4"
)

# 小工具：返回 DataFrame
def q(sql, params=None):
    return pd.read_sql(sqla.text(sql), engine, params=params)


config checked


## 基本统计


In [5]:
# 1) 表行数、时间范围
print(q("SELECT COUNT(*) AS station_count FROM station"))
print(q("SELECT COUNT(*) AS availability_count FROM availability"))
print(q("SELECT MIN(last_update) AS min_time, MAX(last_update) AS max_time FROM availability"))

# 2) availability 中涉及的站点数（与 station 对比）
print(q("SELECT COUNT(DISTINCT number) AS availability_station_count FROM availability"))


   station_count
0            115
   availability_count
0              133269
             min_time            max_time
0 2026-02-03 11:08:39 2026-02-12 20:50:34
   availability_station_count
0                         115


## 最新状态（每个站点一条）


In [6]:
latest_sql = """
SELECT a.*
FROM availability a
JOIN (
    SELECT number, MAX(last_update) AS max_time
    FROM availability
    GROUP BY number
) t ON a.number = t.number AND a.last_update = t.max_time
ORDER BY a.number
LIMIT 10;
"""

with engine.connect() as conn:
    result = conn.execute(sqla.text(latest_sql))
    rows = result.fetchall()

# 简单打印（不依赖 pandas/numpy）
for r in rows:
    print(r)


(1, None, 15, 16, 'OPEN', datetime.datetime(2026, 2, 12, 20, 42, 16), None)
(1, None, 15, 16, 'OPEN', datetime.datetime(2026, 2, 12, 20, 42, 16), None)
(2, None, 17, 3, 'OPEN', datetime.datetime(2026, 2, 12, 20, 47, 16), None)
(2, None, 17, 3, 'OPEN', datetime.datetime(2026, 2, 12, 20, 47, 16), None)
(3, None, 7, 13, 'OPEN', datetime.datetime(2026, 2, 12, 20, 45, 54), None)
(3, None, 7, 13, 'OPEN', datetime.datetime(2026, 2, 12, 20, 45, 54), None)
(4, None, 19, 1, 'OPEN', datetime.datetime(2026, 2, 12, 20, 43, 25), None)
(4, None, 19, 1, 'OPEN', datetime.datetime(2026, 2, 12, 20, 43, 25), None)
(5, None, 16, 24, 'OPEN', datetime.datetime(2026, 2, 12, 20, 50, 34), None)
(6, None, 18, 2, 'OPEN', datetime.datetime(2026, 2, 12, 20, 49, 40), None)


## 每日统计（全站点平均）


In [7]:
daily_sql = '''
SELECT
    DATE(last_update) AS day,
    AVG(available_bikes) AS avg_bikes,
    AVG(available_bike_stands) AS avg_stands,
    COUNT(*) AS records
FROM availability
GROUP BY DATE(last_update)
ORDER BY day;
'''

q(daily_sql).head()


,day,avg_bikes,avg_stands,records
0,2026-02-03,11.7856,19.7473,34146
1,2026-02-04,11.5565,20.0536,65356
2,2026-02-05,11.7692,19.8699,32738
3,2026-02-09,10.8787,20.8319,684
4,2026-02-12,12.2029,19.5971,345


## 每站点每日统计（用于后续特征）


In [8]:
station_daily_sql = '''
SELECT
    number,
    DATE(last_update) AS day,
    AVG(available_bikes) AS avg_bikes,
    AVG(available_bike_stands) AS avg_stands,
    COUNT(*) AS records
FROM availability
GROUP BY number, DATE(last_update)
ORDER BY number, day;
'''

q(station_daily_sql).head()


,number,day,avg_bikes,avg_stands,records
0,1,2026-02-03,9.0467,21.9533,300
1,1,2026-02-04,3.8153,27.1847,574
2,1,2026-02-05,4.7552,26.2448,286
3,1,2026-02-09,26.6667,4.3333,6
4,1,2026-02-12,17.3333,13.6667,3
